# Multimodal Meme Sexism Detection

Arquitectura: **XLM-RoBERTa** + **GatedTextFusion** (separación real OCR/TRANSCRIPTION/REASONING) + dos ramas **CrossModalAttention** (EEG | ET+HR).  
Entrenamiento en dos fases con **SoftLabelLoss** (KLDiv) sobre distribuciones de anotadores.

| Celda | Contenido |
|---|---|
| 1 | Configuración global e imports |
| 2 | `TextEncoder` |
| 3 | `GatedTextFusion` (separación semántica real) + `CrossModalAttentionBranch` |
| 4 | `MultimodalModel` |
| 5 | `SoftLabelLoss` |
| 6 | `MemeDataset` + utilidades |
| 7 | `collate_fn` |
| 8 | Carga de datos y DataLoaders |
| 9 | `evaluate` + `EarlyStopping` |
| 10 | Loop de entrenamiento (`train`) |
| 11 | Lanzar entrenamiento |

## 1 · Configuración global e imports

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import roc_auc_score, f1_score

# ── Constantes ────────────────────────────────────────────────────────────────
TEXT_ENCODER_NAME = "xlm-roberta-base"
MAX_SUBJECTS      = 4
TEXT_DIM          = 768
NUM_CLASSES       = 2        # 0 = no-sexist | 1 = sexist
N_TEXT_SEGMENTS   = 3        # OCR | TRANSCRIPTION | REASONING

PREDS_DIR  = "../data/processed/predictions/"
SAVE_DIR   = "../data/processed/"
DATA_DIR   = "../data/processed/"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 2 · TextEncoder

In [ ]:
class TextEncoder(nn.Module):
    """
    Wrapper de XLM-RoBERTa.
    - freeze=True  : backbone congelado (Fase 1)
    - freeze=False : fine-tune completo (Fase 2)
    """

    def __init__(self, model_name: str = TEXT_ENCODER_NAME, freeze: bool = True):
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.freeze_backbone(freeze)

    def freeze_backbone(self, freeze: bool):
        for p in self.model.parameters():
            p.requires_grad = not freeze

    def get_sequential_output(self, input_ids, attention_mask):
        """Devuelve [B, seq_len, 768]."""
        return self.model(
            input_ids=input_ids, attention_mask=attention_mask
        ).last_hidden_state

## 3 · GatedTextFusion + CrossModalAttentionBranch

### Cambio clave: separación semántica real

El texto se construye como `[OCR] ... </s> [TRANSCRIPTION] ... </s> [REASONING] ...`.
En lugar de dividir la secuencia en franjas posicionales iguales (lo que era completamente arbitrario),
`GatedTextFusion` ahora recibe `sep_positions` — los índices de los tokens `</s>` —
y hace el mean-pool exactamente sobre cada segmento semántico real.

```
input_ids: [CLS] [OCR]...tok_ocr... [SEP] [TRANS]...tok_tr... [SEP] [REAS]...tok_r... [SEP] [PAD]...
                 ↑─────── seg 0 ─────────↑          ↑─────── seg 1 ────────↑         ↑── seg 2 ──↑
```

Si un segmento está vacío (el campo era None/vacío), su representación es el vector cero
y el gate aprende a ignorarlo.

In [ ]:
class GatedTextFusion(nn.Module):
    """
    Fusiona dinámicamente los 3 segmentos semánticos reales:
        OCR  |  TRANSCRIPTION  |  REASONING

    Requiere `sep_positions` [B, n_segments] con los índices de los tokens </s>
    que delimitan cada segmento en la secuencia tokenizada.

    Flujo:
        CLS [B, D]  →  gate_net  →  gates [B, n_seg]  (softmax)
        text_seq    →  mean-pool por segmento semántico real
                    →  seg_stack [B, n_seg, D]
        fused = Σ gates_i * seg_i                      [B, D]

    Si un segmento no existe (todos sus tokens son padding),
    su pool produce ceros y el gate aprende a asignarle peso 0.
    """

    def __init__(self, text_dim: int = TEXT_DIM, n_segments: int = N_TEXT_SEGMENTS, dropout: float = 0.1):
        super().__init__()
        self.n_segments = n_segments
        self.gate_net = nn.Sequential(
            nn.Linear(text_dim, text_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(text_dim // 2, n_segments),
        )

    def forward(
        self,
        text_seq:      torch.Tensor,   # [B, seq_len, D]
        sep_positions: torch.Tensor,   # [B, n_segments]  índices de los </s> por segmento
    ) -> torch.Tensor:                 # [B, D]
        """
        sep_positions[:, i] = índice del token </s> que CIERRA el segmento i.
        El segmento 0 va de la posición 1 (tras CLS) hasta sep_positions[:,0] exclusive.
        El segmento k va de sep_positions[:,k-1]+1 hasta sep_positions[:,k] exclusive.
        """
        B, seq_len, D = text_seq.shape

        # Gate a partir del CLS token
        gates = torch.softmax(self.gate_net(text_seq[:, 0, :]), dim=-1)  # [B, n_seg]

        # Construir máscara booleana por segmento y hacer mean-pool enmascarado
        # positions: [B, seq_len] con índices 0..seq_len-1
        positions = torch.arange(seq_len, device=text_seq.device).unsqueeze(0).expand(B, -1)  # [B, seq_len]

        segments = []
        for i in range(self.n_segments):
            # Inicio del segmento
            if i == 0:
                start = torch.ones(B, device=text_seq.device, dtype=torch.long)  # tras CLS
            else:
                start = sep_positions[:, i - 1] + 1  # tras el </s> anterior

            # Fin del segmento (exclusivo): el </s> que lo cierra
            end = sep_positions[:, i]  # [B]

            # Máscara: True para posiciones dentro del segmento i
            # start y end son [B], positions es [B, seq_len]
            mask = (
                (positions >= start.unsqueeze(1)) &
                (positions <  end.unsqueeze(1))
            )  # [B, seq_len]

            # Mean-pool enmascarado (evita división por cero con clamp)
            mask_f    = mask.unsqueeze(-1).float()           # [B, seq_len, 1]
            seg_sum   = (text_seq * mask_f).sum(dim=1)      # [B, D]
            seg_count = mask_f.sum(dim=1).clamp(min=1.0)    # [B, 1]
            segments.append(seg_sum / seg_count)             # [B, D]

        seg_stack = torch.stack(segments, dim=1)             # [B, n_seg, D]
        return (seg_stack * gates.unsqueeze(-1)).sum(dim=1)  # [B, D]


class CrossModalAttentionBranch(nn.Module):
    """
    Rama fisiológica independiente (EEG o ET+HR).

    physio [B, MAX_S, physio_dim]
        ↓  MLP Projection  →  [B, MAX_S, D]
        ↓  Cross-Attention  Q=physio, K=V=text_seq
        ↓  MLP Importancia + Softmax  (mask padding → -inf)
    [B, D]
    """

    def __init__(self, physio_dim: int, text_dim: int = TEXT_DIM,
                 num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        self.physio_proj = nn.Sequential(
            nn.Linear(physio_dim, text_dim),
            nn.LayerNorm(text_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=text_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True,
        )
        self.attn_norm      = nn.LayerNorm(text_dim)
        self.importance_mlp = nn.Sequential(
            nn.Linear(text_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, physio_seq: torch.Tensor, text_seq: torch.Tensor,
                physio_mask: torch.Tensor) -> torch.Tensor:
        """
        physio_seq  : [B, MAX_S, physio_dim]
        text_seq    : [B, seq_len, D]
        physio_mask : [B, MAX_S]  True=real, False=padding
        Returns     : [B, D]
        """
        p           = self.physio_proj(physio_seq)
        attn_out, _ = self.cross_attn(query=p, key=text_seq, value=text_seq)
        p_attended  = self.attn_norm(p + attn_out) * physio_mask.unsqueeze(-1).float()

        scores  = self.importance_mlp(p_attended)
        scores  = scores.masked_fill(~physio_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(scores, dim=1)
        return (p_attended * weights).sum(dim=1)

## 4 · MultimodalModel

In [ ]:
class MultimodalModel(nn.Module):
    """
    Arquitectura completa:

        Text (OCR </s> TRANSCRIPTION </s> REASONING)
            ↓  XLM-RoBERTa  →  text_seq [B, seq_len, 768]
            ↓  GatedTextFusion(sep_positions)  →  text_fused  [B, 768]
                 (mean-pool por segmento semántico real, no por franjas)

        EEG   → CrossModalAttentionBranch(text_seq) →  eeg_ctx   [B, 768]
        ET+HR → CrossModalAttentionBranch(text_seq) →  ethr_ctx  [B, 768]

        [cls_token ‖ text_fused ‖ eeg_ctx ‖ ethr_ctx]  →  [B, 3072]
            ↓  MLP clasificador
        logits [B, 2]
    """

    def __init__(
        self,
        eeg_dim:         int,
        et_hr_dim:       int,
        text_dim:        int   = TEXT_DIM,
        num_heads:       int   = 8,
        n_text_segments: int   = N_TEXT_SEGMENTS,
        freeze_backbone: bool  = True,
        dropout:         float = 0.1,
    ):
        super().__init__()
        self.text_encoder = TextEncoder(freeze=freeze_backbone)
        self.text_gate    = GatedTextFusion(text_dim, n_text_segments, dropout)
        self.eeg_branch   = CrossModalAttentionBranch(eeg_dim,   text_dim, num_heads, dropout)
        self.et_hr_branch = CrossModalAttentionBranch(et_hr_dim, text_dim, num_heads, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(text_dim * 4, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(64, NUM_CLASSES),
        )

    def forward(
        self,
        input_ids:      torch.Tensor,   # [B, seq_len]
        attention_mask: torch.Tensor,   # [B, seq_len]
        sep_positions:  torch.Tensor,   # [B, N_TEXT_SEGMENTS]  ← NUEVO
        eeg:            torch.Tensor,   # [B, MAX_S, eeg_dim]
        eeg_mask:       torch.Tensor,   # [B, MAX_S]
        et_hr:          torch.Tensor,   # [B, MAX_S, et_hr_dim]
        et_hr_mask:     torch.Tensor,   # [B, MAX_S]
    ) -> torch.Tensor:                  # [B, 2]
        text_seq   = self.text_encoder.get_sequential_output(input_ids, attention_mask)
        cls_token  = text_seq[:, 0, :]
        text_fused = self.text_gate(text_seq, sep_positions)   # separación semántica real
        eeg_ctx    = self.eeg_branch(eeg,   text_seq, eeg_mask)
        et_hr_ctx  = self.et_hr_branch(et_hr, text_seq, et_hr_mask)
        fused      = torch.cat([cls_token, text_fused, eeg_ctx, et_hr_ctx], dim=1)
        return self.classifier(fused)

## 5 · SoftLabelLoss

KL-Divergence sobre distribuciones de anotadores en lugar de CrossEntropy con hard labels.  
Un meme anotado [1,1,1,0] genera soft_label=[0.25, 0.75]; el modelo es penalizado
*proporcionalmente* a su desacuerdo con esa distribución, no de forma binaria.

In [ ]:
class SoftLabelLoss(nn.Module):
    """KLDiv con label smoothing opcional sobre la distribución target."""
 
    def __init__(self, smoothing: float = 0.1, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.smoothing   = smoothing
        self.num_classes = num_classes
        self.kl          = nn.KLDivLoss(reduction="batchmean")
 
    def forward(self, logits: torch.Tensor, soft_targets: torch.Tensor) -> torch.Tensor:
        uniform  = torch.full_like(soft_targets, 1.0 / self.num_classes)
        smoothed = (1 - self.smoothing) * soft_targets + self.smoothing * uniform
        return self.kl(F.log_softmax(logits, dim=-1), smoothed)

## 6 · MemeDataset + utilidades

**Cambio clave:** `__getitem__` ahora calcula y devuelve `sep_positions` —
los índices reales de los tokens `</s>` que separan OCR / TRANSCRIPTION / REASONING
en la secuencia tokenizada.

**Estrategia de búsqueda de `</s>`:**
- XLM-RoBERTa usa el token `</s>` (id=2) como separador entre segmentos.
- El texto se construye como `"[OCR] ... </s> [TRANSCRIPTION] ... </s> [REASONING] ..."`
  y el tokenizador añade automáticamente `<s>` al inicio y `</s>` al final.
- Los 3 primeros tokens `</s>` encontrados de izquierda a derecha corresponden
  al cierre de cada uno de los 3 segmentos.
- Si un segmento estaba vacío (campo None/""), no genera texto y por tanto
  no genera un `</s>` interior; en ese caso se usa un fallback (CLS+1)
  para que el mean-pool resulte en vector cero.

In [ ]:
def build_segment_encoding(
    tokenizer,
    parts: list,           # lista de strings, uno por segmento (puede estar vacío)
    max_length: int = 384,
) -> tuple:
    """
    Tokeniza cada segmento por separado, los concatena manualmente con el
    token especial </s> (id=2) del tokenizador, y devuelve:
        input_ids      : LongTensor [max_length]
        attention_mask : LongTensor [max_length]
        seg_ends       : LongTensor [n_segments]  — índice del </s> que cierra cada segmento

    Estructura resultante (XLM-RoBERTa):
        <s>  OCR_tokens...  </s>  TRANS_tokens...  </s>  REAS_tokens...  </s>  <PAD>...
         0        1..e0       e0       e0+1..e1      e1        e1+1..e2    e2

    De este modo los </s> son siempre tokens especiales reales (id=2),
    no texto libre tokenizado como caracteres sueltos.
    Si un segmento está vacío, contribuye 0 tokens de contenido y su </s>
    queda contiguo al </s> anterior → rango vacío → mean-pool = 0.
    """
    CLS_ID = tokenizer.cls_token_id   # 0 en XLM-RoBERTa
    SEP_ID = tokenizer.sep_token_id   # 2 en XLM-RoBERTa
    PAD_ID = tokenizer.pad_token_id   # 1 en XLM-RoBERTa

    # Tokenizar cada parte SIN tokens especiales, para controlar la concatenación
    token_ids_per_seg = []
    for part in parts:
        if part and part.strip():
            ids = tokenizer(
                part,
                add_special_tokens=False,   # sin <s> ni </s> automáticos
            )["input_ids"]
        else:
            ids = []  # segmento vacío
        token_ids_per_seg.append(ids)

    # Construir secuencia: <s>  [seg0] </s>  [seg1] </s>  [seg2] </s>
    # Truncar repartiendo el presupuesto de tokens entre los segmentos no vacíos
    overhead    = 1 + len(parts)          # <s> + N × </s>
    budget      = max_length - overhead   # tokens de contenido disponibles
    n_nonempty  = sum(1 for ids in token_ids_per_seg if ids)
    per_seg_max = budget // max(n_nonempty, 1)

    seq        = [CLS_ID]
    seg_ends   = []
    for ids in token_ids_per_seg:
        truncated = ids[:per_seg_max]
        seq.extend(truncated)
        seq.append(SEP_ID)
        seg_ends.append(len(seq) - 1)    # índice del </s> que cierra este segmento

    # Padding hasta max_length
    attn   = [1] * len(seq) + [0] * (max_length - len(seq))
    seq    = seq             + [PAD_ID] * (max_length - len(seq))

    return (
        torch.tensor(seq,      dtype=torch.long),
        torch.tensor(attn,     dtype=torch.long),
        torch.tensor(seg_ends, dtype=torch.long),   # [n_segments]
    )


def pad_subjects(subject_list: list, max_subjects: int, feat_dim: int):
    """Lista de vectores → tensor [max_subjects, feat_dim] + máscara booleana."""
    tensor = torch.zeros(max_subjects, feat_dim)
    mask   = torch.zeros(max_subjects, dtype=torch.bool)
    for i, s in enumerate(subject_list[:max_subjects]):
        tensor[i] = torch.tensor(s, dtype=torch.float)
        mask[i]   = True
    return tensor, mask


def votes_to_soft_label(votes: list, num_classes: int = NUM_CLASSES) -> torch.Tensor:
    """[1,1,1,0]  →  tensor([0.25, 0.75])"""
    counts = torch.zeros(num_classes)
    for v in votes:
        counts[v] += 1
    return counts / counts.sum()


def parse_qwen_label(qwen_label) -> int:
    """Convierte qwen_label (YES/NO/otro) a {1, 0, -1} para fallback."""
    if qwen_label is None:
        return -1
    label = str(qwen_label).strip().upper()
    if label == "YES":
        return 1
    if label == "NO":
        return 0
    return -1


class MemeDataset(Dataset):
    """
    Campos esperados por sample:
        qwen_ocr           : str | None
        qwen_transcription : str | None
        qwen_reasoning     : str | None
        qwen_label         : str (YES/NO) opcional para fallback
        physio             : {"EEG": [...], "ET": [...], "HR": [...]}
        votes              : list[int]   (preferido: [0,1,1,0])
        label              : int         (fallback si no hay votes)

    Novedades vs versión anterior:
        - Devuelve `sep_positions` [N_TEXT_SEGMENTS] con los índices reales
          de los tokens </s> que delimitan cada segmento semántico.
        - Puede devolver `qwen_pred` para fallback por incertidumbre.
    """

    def __init__(self, data, tokenizer, eeg_dim=None, et_hr_dim=None, max_length=384, include_qwen=False):
        self.data         = data
        self.tokenizer    = tokenizer
        self.max_length   = max_length
        self.include_qwen = include_qwen

        first = data[0]["physio"]
        eeg_s = first.get("EEG", [])
        et_s  = first.get("ET",  [])
        hr_s  = first.get("HR",  [])

        self.eeg_dim   = eeg_dim   or (len(eeg_s[0]) if eeg_s else 1)
        et_dim         = len(et_s[0]) if et_s else 0
        hr_dim         = len(hr_s[0]) if hr_s else 0
        self.et_hr_dim = et_hr_dim or (et_dim + hr_dim) or 1

        print(f"[MemeDataset] EEG_DIM={self.eeg_dim} | ET_HR_DIM={self.et_hr_dim} | n={len(data)} | include_qwen={self.include_qwen}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        # ── 1. Texto: tokenización por segmento, con separadores reales ───
        # Siempre pasamos los 3 segmentos en orden fijo (OCR / TRANS / REAS).
        # Si un campo está ausente, se pasa string vacío → rango vacío → cero.
        parts = [
            f"[OCR] {sample['qwen_ocr']}"               if sample.get("qwen_ocr")           else "",
            f"[TRANSCRIPTION] {sample['qwen_transcription']}" if sample.get("qwen_transcription") else "",
            f"[REASONING] {sample['qwen_reasoning']}"   if sample.get("qwen_reasoning")     else "",
        ]

        input_ids, attention_mask, sep_positions = build_segment_encoding(
            self.tokenizer, parts, self.max_length
        )
        # input_ids     : [max_length]       — tokens con </s> reales en posiciones conocidas
        # attention_mask: [max_length]
        # sep_positions : [N_TEXT_SEGMENTS]  — índice del </s> que cierra cada segmento

        # ── 2. Fisiología ─────────────────────────────────────────────────
        physio  = sample.get("physio", {})
        eeg_s   = physio.get("EEG", [])
        et_s    = physio.get("ET",  [])
        hr_s    = physio.get("HR",  [])

        n       = max(len(et_s), len(hr_s))
        et_dim  = len(et_s[0]) if et_s else 0
        hr_dim  = len(hr_s[0]) if hr_s else 0
        et_hr   = [
            (et_s[i] if i < len(et_s) else [0.0]*et_dim) +
            (hr_s[i] if i < len(hr_s) else [0.0]*hr_dim)
            for i in range(n)
        ]

        eeg_seq,   eeg_mask   = pad_subjects(eeg_s, MAX_SUBJECTS, self.eeg_dim)
        et_hr_seq, et_hr_mask = pad_subjects(et_hr, MAX_SUBJECTS, self.et_hr_dim)

        # ── 4. Soft label ─────────────────────────────────────────────────
        soft_label = votes_to_soft_label(sample["label"])

        # qwen_pred: 1=YES, 0=NO, -1=desconocido/no disponible
        qwen_pred = parse_qwen_label(sample.get("qwen_label")) if self.include_qwen else -1

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "sep_positions":  sep_positions,   # [N_TEXT_SEGMENTS]
            "eeg":            eeg_seq,
            "eeg_mask":       eeg_mask,
            "et_hr":          et_hr_seq,
            "et_hr_mask":     et_hr_mask,
            "soft_label":     soft_label,      # [2]
            "qwen_pred":      torch.tensor(qwen_pred, dtype=torch.long),
        }

## 7 · collate_fn

In [ ]:
def collate_fn(batch):
    return {
        "input_ids":      torch.stack([b["input_ids"]      for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "sep_positions":  torch.stack([b["sep_positions"]  for b in batch]),  # [B, N_TEXT_SEGMENTS]
        "eeg":            torch.stack([b["eeg"]            for b in batch]),
        "eeg_mask":       torch.stack([b["eeg_mask"]       for b in batch]),
        "et_hr":          torch.stack([b["et_hr"]          for b in batch]),
        "et_hr_mask":     torch.stack([b["et_hr_mask"]     for b in batch]),
        "soft_label":     torch.stack([b["soft_label"]     for b in batch]),   # [B, 2]
        "qwen_pred":      torch.stack([b["qwen_pred"]      for b in batch]),   # [B] {-1,0,1}
    }

## 8 · Carga de datos y DataLoaders

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TEXT_ENCODER_NAME)

with open(os.path.join(DATA_DIR, "train.json"), encoding="utf-8") as f:
    train_data_all = json.load(f)

with open(os.path.join(DATA_DIR, "val.json"), encoding="utf-8") as f:
    val_data = json.load(f)

# Train only with English samples
train_data = [s for s in train_data_all if s.get("lang") == "es"]

print(f"Train total: {len(train_data_all)} | Train EN-only: {len(train_data)} | Val(all): {len(val_data)}")

train_dataset = MemeDataset(train_data, tokenizer)
val_dataset   = MemeDataset(val_data,   tokenizer,
                            eeg_dim=train_dataset.eeg_dim,
                            et_hr_dim=train_dataset.et_hr_dim)

eeg_dim   = train_dataset.eeg_dim
et_hr_dim = train_dataset.et_hr_dim

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False,
                          collate_fn=collate_fn, num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 9 · evaluate + EarlyStopping

In [ ]:
def apply_qwen_fallback(
    probs: np.ndarray,
    qwen_preds: np.ndarray,
    decision_threshold: float = 0.5,
    uncertainty_margin: float = 0.0,
    fallback_to_qwen: bool = False,
):
    """
    Convierte probs a predicción binaria y, si corresponde, aplica fallback a Qwen.

    - uncertain: |prob - decision_threshold| <= uncertainty_margin
    - fallback solo si qwen_pred ∈ {0,1}
    """
    probs = np.asarray(probs, dtype=float)
    qwen_preds = np.asarray(qwen_preds, dtype=int)

    preds = (probs >= decision_threshold).astype(int)
    fallback_mask = np.zeros_like(preds, dtype=bool)

    if not fallback_to_qwen or uncertainty_margin <= 0:
        return preds, fallback_mask

    uncertain_mask = np.abs(probs - decision_threshold) <= uncertainty_margin
    qwen_available = np.isin(qwen_preds, [0, 1])
    fallback_mask = uncertain_mask & qwen_available

    preds = preds.copy()
    preds[fallback_mask] = qwen_preds[fallback_mask]
    return preds, fallback_mask


def compute_classification_metrics(labels: np.ndarray, probs: np.ndarray, preds: np.ndarray):
    """Métricas robustas para casos degenerados (una sola clase)."""
    labels = np.asarray(labels, dtype=int)
    probs = np.asarray(probs, dtype=float)
    preds = np.asarray(preds, dtype=int)

    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")

    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    f1_yes = f1_score(labels, preds, pos_label=1, average="binary", zero_division=0)
    return auc, f1_macro, f1_yes


def _downsample_candidates(values: np.ndarray, max_candidates: int = 400) -> np.ndarray:
    """Reduce candidatos si hay demasiados, preservando cobertura por cuantiles."""
    values = np.asarray(values, dtype=float)
    if len(values) <= max_candidates:
        return values
    q = np.linspace(0.0, 1.0, max_candidates)
    return np.unique(np.quantile(values, q))


def build_midpoint_threshold_candidates(
    probs: np.ndarray,
    min_value: float = 0.0,
    max_value: float = 1.0,
    include_default: bool = True,
    max_candidates: int = 400,
) -> np.ndarray:
    """
    Candidatos de threshold = puntos medios entre probabilidades únicas ordenadas.
    Ej: probs [0.1, 0.2, 0.4] -> thresholds [0.15, 0.30]
    """
    probs = np.asarray(probs, dtype=float)
    vals = np.unique(np.clip(probs, min_value, max_value))

    if len(vals) < 2:
        cands = np.array([0.5], dtype=float)
    else:
        cands = (vals[:-1] + vals[1:]) / 2.0

    if include_default:
        cands = np.unique(np.concatenate([cands, np.array([0.0, 0.5, 1.0], dtype=float)]))

    cands = np.clip(cands, min_value, max_value)
    cands = _downsample_candidates(cands, max_candidates=max_candidates)
    return np.sort(np.unique(cands))


def build_midpoint_uncertainty_candidates(
    probs: np.ndarray,
    threshold: float,
    max_margin: float = 0.5,
    include_zero: bool = True,
    max_candidates: int = 400,
) -> np.ndarray:
    """
    Candidatos de margen = puntos medios entre distancias |prob-threshold| únicas.
    """
    probs = np.asarray(probs, dtype=float)
    d = np.unique(np.clip(np.abs(probs - float(threshold)), 0.0, max_margin))

    if len(d) < 2:
        cands = np.array([0.0], dtype=float)
    else:
        cands = (d[:-1] + d[1:]) / 2.0

    if include_zero:
        cands = np.unique(np.concatenate([np.array([0.0], dtype=float), cands]))

    cands = np.clip(cands, 0.0, max_margin)
    cands = _downsample_candidates(cands, max_candidates=max_candidates)
    return np.sort(np.unique(cands))


def search_qwen_fallback_sweet_spot(
    probs: np.ndarray,
    labels: np.ndarray,
    qwen_preds: np.ndarray,
    threshold_grid=None,
    uncertainty_grid=None,
    sort_by: str = "f1_macro",
    top_k: int = 10,
    fallback_to_qwen: bool = True,
    fixed_threshold: float = None,
):
    """
    Busca la mejor combinación (threshold, uncertainty_margin) usando probs ya calculadas.

    Modos:
    - Ajuste de umbral sin fallback: fallback_to_qwen=False (um=0)
    - Ajuste de fallback: fixed_threshold=<th_opt>, fallback_to_qwen=True
    - Si no pasas grids, usa candidatos inteligentes (midpoints de probs).
    """
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels, dtype=int)
    qwen_preds = np.asarray(qwen_preds, dtype=int)

    if fixed_threshold is not None:
        thresholds = np.array([float(fixed_threshold)], dtype=float)
    else:
        if threshold_grid is None:
            thresholds = build_midpoint_threshold_candidates(probs)
        else:
            thresholds = np.asarray(threshold_grid, dtype=float)

    results = []
    for th in thresholds:
        if not fallback_to_qwen:
            uncertainties = np.array([0.0], dtype=float)
        else:
            if uncertainty_grid is None:
                uncertainties = build_midpoint_uncertainty_candidates(probs, threshold=float(th))
            else:
                uncertainties = np.asarray(uncertainty_grid, dtype=float)

        for um in uncertainties:
            preds, fallback_mask = apply_qwen_fallback(
                probs=probs,
                qwen_preds=qwen_preds,
                decision_threshold=float(th),
                uncertainty_margin=float(um),
                fallback_to_qwen=fallback_to_qwen,
            )
            auc, f1_macro, f1_yes = compute_classification_metrics(labels, probs, preds)
            results.append({
                "threshold": float(th),
                "uncertainty_margin": float(um),
                "auc": float(auc) if not np.isnan(auc) else float("nan"),
                "f1_macro": float(f1_macro),
                "f1_yes": float(f1_yes),
                "fallback_to_qwen": bool(fallback_to_qwen),
                "fallback_count": int(fallback_mask.sum()),
                "fallback_rate": float(fallback_mask.mean()),
            })

    valid_sort_keys = {"f1_macro", "f1_yes", "auc"}
    if sort_by not in valid_sort_keys:
        sort_by = "f1_macro"

    results_sorted = sorted(results, key=lambda x: (x[sort_by], x["f1_yes"]), reverse=True)
    best_result = results_sorted[0]
    return best_result, results_sorted[:top_k]


def staged_threshold_tuning(
    probs: np.ndarray,
    labels: np.ndarray,
    qwen_preds: np.ndarray,
    threshold_grid=None,
    uncertainty_grid=None,
    sort_by: str = "f1_macro",
):
    """
    Pipeline en 3 pasos:
      1) Baseline: th=0.5, sin fallback
      2) Ajuste de threshold: sin fallback
      3) Ajuste de uncertainty: con fallback, centrado en th ajustado

    Si no se pasan grids, usa candidatos inteligentes por midpoints.
    """
    probs = np.asarray(probs, dtype=float)
    labels = np.asarray(labels, dtype=int)
    qwen_preds = np.asarray(qwen_preds, dtype=int)

    # 1) baseline
    baseline_preds, baseline_fb = apply_qwen_fallback(
        probs=probs,
        qwen_preds=qwen_preds,
        decision_threshold=0.5,
        uncertainty_margin=0.0,
        fallback_to_qwen=False,
    )
    b_auc, b_f1_macro, b_f1_yes = compute_classification_metrics(labels, probs, baseline_preds)
    baseline = {
        "threshold": 0.5,
        "uncertainty_margin": 0.0,
        "fallback_to_qwen": False,
        "auc": float(b_auc) if not np.isnan(b_auc) else float("nan"),
        "f1_macro": float(b_f1_macro),
        "f1_yes": float(b_f1_yes),
        "fallback_count": int(baseline_fb.sum()),
        "fallback_rate": float(baseline_fb.mean()),
    }

    # 2) ajustar threshold sin fallback
    best_no_fb, top_no_fb = search_qwen_fallback_sweet_spot(
        probs=probs,
        labels=labels,
        qwen_preds=qwen_preds,
        threshold_grid=threshold_grid,
        uncertainty_grid=[0.0],
        sort_by=sort_by,
        top_k=10,
        fallback_to_qwen=False,
    )

    # 3) con threshold fijo, ajustar incertidumbre con fallback
    best_fb, top_fb = search_qwen_fallback_sweet_spot(
        probs=probs,
        labels=labels,
        qwen_preds=qwen_preds,
        threshold_grid=[best_no_fb["threshold"]],
        uncertainty_grid=uncertainty_grid,
        sort_by=sort_by,
        top_k=10,
        fallback_to_qwen=True,
        fixed_threshold=best_no_fb["threshold"],
    )

    return {
        "baseline": baseline,
        "best_no_fallback": best_no_fb,
        "top_no_fallback": top_no_fb,
        "best_with_fallback": best_fb,
        "top_with_fallback": top_fb,
    }


def evaluate(
    model,
    criterion,
    loader,
    device,
    save_path=None,
    decision_threshold: float = 0.5,
    uncertainty_margin: float = 0.0,
    fallback_to_qwen: bool = False,
    return_outputs: bool = False,
):
    """
    Evalúa sobre `loader` y devuelve métricas.

    Opciones nuevas:
    - `decision_threshold`: umbral para convertir prob→clase.
    - `uncertainty_margin`: zona de incertidumbre alrededor del threshold.
    - `fallback_to_qwen`: si True, usa qwen_pred en muestras inciertas.
    - `return_outputs`: si True, devuelve también probs/labels/qwen para sweep.
    """
    model.eval()
    all_probs, all_labels, all_qwen, total_loss = [], [], [], 0.0

    with torch.no_grad():
        for batch in loader:
            logits = model(
                input_ids      = batch["input_ids"].to(device),
                attention_mask = batch["attention_mask"].to(device),
                sep_positions  = batch["sep_positions"].to(device),
                eeg            = batch["eeg"].to(device),
                eeg_mask       = batch["eeg_mask"].to(device),
                et_hr          = batch["et_hr"].to(device),
                et_hr_mask     = batch["et_hr_mask"].to(device),
            )
            soft_labels = batch["soft_label"].to(device)
            total_loss += criterion(logits, soft_labels).item()

            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            labels = batch["soft_label"].argmax(dim=1).cpu().numpy()

            if "qwen_pred" in batch:
                qwen_preds = batch["qwen_pred"].cpu().numpy()
            else:
                qwen_preds = np.full_like(labels, fill_value=-1, dtype=int)

            all_probs.extend(probs)
            all_labels.extend(labels)
            all_qwen.extend(qwen_preds)

    all_probs = np.array(all_probs, dtype=float)
    all_labels = np.array(all_labels, dtype=int)
    all_qwen = np.array(all_qwen, dtype=int)

    preds, fallback_mask = apply_qwen_fallback(
        probs=all_probs,
        qwen_preds=all_qwen,
        decision_threshold=decision_threshold,
        uncertainty_margin=uncertainty_margin,
        fallback_to_qwen=fallback_to_qwen,
    )
    auc, f1_macro, f1_yes = compute_classification_metrics(all_labels, all_probs, preds)

    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        new_metrics = {
            "auc": round(float(auc), 4) if not np.isnan(auc) else None,
            "f1_macro": round(float(f1_macro), 4),
            "f1_yes": round(float(f1_yes), 4),
            "threshold": float(decision_threshold),
            "uncertainty_margin": float(uncertainty_margin),
            "fallback_to_qwen": bool(fallback_to_qwen),
            "fallback_count": int(fallback_mask.sum()),
            "fallback_rate": round(float(fallback_mask.mean()), 4),
        }

        save = True
        if os.path.exists(save_path):
            with open(save_path) as f:
                old_f1 = json.load(f).get("metrics", {}).get("f1_macro", -1)
                save = new_metrics["f1_macro"] > old_f1

        if save:
            print("  [evaluate] Guardando predicciones...")
            with open(save_path, "w") as f:
                json.dump({
                    "predictions": [
                        {
                            "label": int(l),
                            "pred": int(p),
                            "prob": float(prob),
                            "qwen_pred": int(q),
                            "used_qwen_fallback": bool(use_fb),
                        }
                        for l, p, prob, q, use_fb in zip(all_labels, preds, all_probs, all_qwen, fallback_mask)
                    ],
                    "metrics": new_metrics,
                }, f, indent=2)

    val_loss = total_loss / len(loader)

    if return_outputs:
        return {
            "loss": val_loss,
            "auc": auc,
            "f1_macro": f1_macro,
            "f1_yes": f1_yes,
            "preds": preds,
            "probs": all_probs,
            "labels": all_labels,
            "qwen_preds": all_qwen,
            "fallback_mask": fallback_mask,
        }

    return val_loss, auc, f1_macro, f1_yes


class EarlyStopping:
    """
    Detiene el entrenamiento si F1-macro no mejora en `patience` épocas.
    Guarda automáticamente el mejor checkpoint.
    """

    def __init__(self, patience: int = 4, min_delta: float = 1e-4, save_path: str = "best_model.pt"):
        self.patience  = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.best_f1   = -1.0
        self.counter   = 0

    def step(self, f1: float, model: nn.Module) -> bool:
        """Devuelve True si hay que parar."""
        if f1 > self.best_f1 + self.min_delta:
            self.best_f1 = f1
            self.counter = 0
            torch.save(model.state_dict(), self.save_path)
            return False
        self.counter += 1
        return self.counter >= self.patience

## 10 · Función `train`

**Fase 1** — backbone congelado; solo se entrenan gate, ramas fisiológicas y clasificador.  
**Fase 2** — fine-tune completo con LR discriminativos por capa del encoder.

In [ ]:
def _forward_batch(model, batch, device):
    """Helper para llamar al modelo con el nuevo argumento sep_positions."""
    return model(
        input_ids      = batch["input_ids"].to(device),
        attention_mask = batch["attention_mask"].to(device),
        sep_positions  = batch["sep_positions"].to(device),
        eeg            = batch["eeg"].to(device),
        eeg_mask       = batch["eeg_mask"].to(device),
        et_hr          = batch["et_hr"].to(device),
        et_hr_mask     = batch["et_hr_mask"].to(device),
    )


def train(
    train_data,
    loader,
    val_loader,
    eeg_dim:           int,
    et_hr_dim:         int,
    text_encoder_name: str   = TEXT_ENCODER_NAME,
    save_dir:          str   = SAVE_DIR,
    phase1_epochs:     int   = 5,
    phase2_epochs:     int   = 10,
    es_patience:       int   = 4,
):
    os.makedirs(save_dir, exist_ok=True)
    json_path  = os.path.join(PREDS_DIR, f"{text_encoder_name}.json")
    model_path = os.path.join(save_dir,  f"{text_encoder_name}.pt")

    model = MultimodalModel(
        eeg_dim=eeg_dim, et_hr_dim=et_hr_dim,
        text_dim=768, num_heads=8, freeze_backbone=True,
    ).to(device)

    criterion = SoftLabelLoss()

    # ── Fase 1: backbone congelado ────────────────────────────────────────
    print("\n=== FASE 1: backbone congelado ===\n")

    phase1_lr = 1e-5
    params1    = [p for n, p in model.named_parameters() if "text_encoder" not in n and p.requires_grad]
    optimizer1 = torch.optim.AdamW(params1, lr=phase1_lr, weight_decay=0.05)
    scheduler1 = CosineAnnealingLR(optimizer1, T_max=phase1_epochs, eta_min=1e-6)
    phase1_model_path = os.path.join(save_dir, f"{text_encoder_name}.phase1_best.pt")
    best_f1_phase1 = -1.0

    for epoch in range(phase1_epochs):
        model.train()
        pbar = tqdm(loader, desc=f"Ph1 {epoch+1}/{phase1_epochs}")
        for batch in pbar:
            optimizer1.zero_grad()
            logits = _forward_batch(model, batch, device)
            loss   = criterion(logits, batch["soft_label"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer1.step()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        scheduler1.step()

        val_loss, auc, f1, f1_yes = evaluate(model, criterion, val_loader, device, json_path)
        marker = " ← best" if f1 > best_f1_phase1 else ""
        if f1 > best_f1_phase1:
            best_f1_phase1 = f1
            torch.save(model.state_dict(), phase1_model_path)
        print(f"  → AUC={auc:.4f}  F1={f1:.4f}  F1_yes={f1_yes:.4f}  loss={val_loss:.4f}{marker}")

    if os.path.exists(phase1_model_path):
        model.load_state_dict(torch.load(phase1_model_path, map_location=device))
        print(f"\n✅ Mejor modelo de Fase 1 cargado  (F1={best_f1_phase1:.4f})")

    # ── Fase 2: fine-tune con LR discriminativos ──────────────────────────
    print("\n=== FASE 2: fine-tune con LR discriminativos ===\n")

    model.text_encoder.freeze_backbone(False)
    enc_layers = list(model.text_encoder.model.encoder.layer)
    n          = len(enc_layers)

    param_groups = [
        {"params": model.text_encoder.model.embeddings.parameters(), "lr": 1e-6},
        *[{"params": l.parameters(), "lr": 1e-6} for l in enc_layers[:n//2]],
        *[{"params": l.parameters(), "lr": 5e-6} for l in enc_layers[n//2:]],
        {"params": [p for name, p in model.named_parameters()
                    if "text_encoder" not in name], "lr": 2e-5},
    ]
    optimizer2 = torch.optim.AdamW(param_groups, weight_decay=0.05)
    scheduler2 = CosineAnnealingLR(optimizer2, T_max=phase2_epochs, eta_min=1e-8)
    early_stop = EarlyStopping(patience=es_patience, save_path=model_path)

    phase2_model_path = os.path.join(save_dir, f"{text_encoder_name}.phase2_best.pt")
    best_f1_phase2 = -1.0
    for epoch in range(phase2_epochs):
        model.train()
        pbar = tqdm(loader, desc=f"Ph2 {epoch+1}/{phase2_epochs}")
        for batch in pbar:
            optimizer2.zero_grad()
            logits = _forward_batch(model, batch, device)
            loss   = criterion(logits, batch["soft_label"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer2.step()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        scheduler2.step()

        val_loss, auc, f1, f1_yes = evaluate(model, criterion, val_loader, device, json_path)
        marker  = " ← best" if f1 > best_f1_phase2 else ""
        if f1 > best_f1_phase2:
            best_f1_phase2 = f1
            torch.save(model.state_dict(), phase2_model_path)
        print(f"  → AUC={auc:.4f}  F1={f1:.4f}  F1_yes={f1_yes:.4f}  loss={val_loss:.4f}{marker}")

        if early_stop.step(f1, model):
            print(f"  [EarlyStopping] Sin mejora en {es_patience} épocas. Parando.")
            break

    if os.path.exists(phase2_model_path):
        model.load_state_dict(torch.load(phase2_model_path, map_location=device))
        print(f"\n✅ Mejor modelo de Fase 2 cargado  (F1={best_f1_phase2:.4f})")
    elif os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"\n✅ Mejor modelo cargado  (F1={early_stop.best_f1:.4f})")

    return model

## 11 · Lanzar entrenamiento

In [ ]:
model = train(
    train_data        = train_data,
    loader            = train_loader,
    val_loader        = val_loader,
    eeg_dim           = eeg_dim,
    et_hr_dim         = et_hr_dim,
    text_encoder_name = TEXT_ENCODER_NAME,
    phase1_epochs=10,
    phase2_epochs=20,
    es_patience=4
)

# 12 Evaluar

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12 · Evaluación sobre test (pipeline por etapas, candidatos inteligentes)
# ══════════════════════════════════════════════════════════════════════════════

with open(os.path.join(DATA_DIR, "test.json"), encoding="utf-8") as f:
    test_data = json.load(f)

test_dataset = MemeDataset(
    test_data, tokenizer,
    eeg_dim=eeg_dim, et_hr_dim=et_hr_dim,
    include_qwen=True,  # habilita qwen_pred para fallback
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_fn, num_workers=4, pin_memory=True,
 )

test_save_path = os.path.join(PREDS_DIR, f"{TEXT_ENCODER_NAME}_test.json")

# 1) Obtener probs del modelo sin fallback y con threshold fijo en 0.5
eval_base = evaluate(
    model=model,
    criterion=SoftLabelLoss(),
    loader=test_loader,
    device=device,
    save_path=None,
    decision_threshold=0.50,
    uncertainty_margin=0.0,
    fallback_to_qwen=False,
    return_outputs=True,
 )

# 2) Ajuste por etapas (sin grids fijos):
#    a) baseline 0.5 sin fallback
#    b) mejor threshold por midpoints de probs (sin fallback)
#    c) mejor uncertainty por midpoints de distancias al threshold ajustado
stage = staged_threshold_tuning(
    probs=eval_base["probs"],
    labels=eval_base["labels"],
    qwen_preds=eval_base["qwen_preds"],
    threshold_grid=None,
    uncertainty_grid=None,
    sort_by="f1_macro",
)

baseline_cfg = stage["baseline"]
best_no_fb = stage["best_no_fallback"]
best_fb = stage["best_with_fallback"]

print("\n" + "═" * 68)
print("  STAGE 1 · Baseline (threshold=0.50, sin fallback)")
print("═" * 68)
print(f"  AUC      : {baseline_cfg['auc']:.4f}" if not np.isnan(baseline_cfg["auc"]) else "  AUC      : nan")
print(f"  F1_macro : {baseline_cfg['f1_macro']:.4f}")
print(f"  F1_yes   : {baseline_cfg['f1_yes']:.4f}")

print("\n" + "═" * 68)
print("  STAGE 2 · Mejor threshold (sin fallback, midpoints)")
print("═" * 68)
print(f"  threshold          : {best_no_fb['threshold']:.6f}")
print(f"  uncertainty_margin : {best_no_fb['uncertainty_margin']:.6f}")
print(f"  AUC                : {best_no_fb['auc']:.4f}" if not np.isnan(best_no_fb["auc"]) else "  AUC                : nan")
print(f"  F1_macro           : {best_no_fb['f1_macro']:.4f}")
print(f"  F1_yes             : {best_no_fb['f1_yes']:.4f}")

print("\n" + "═" * 68)
print("  STAGE 3 · Mejor fallback uncertainty (midpoints, centrado en threshold ajustado)")
print("═" * 68)
print(f"  threshold          : {best_fb['threshold']:.6f}")
print(f"  uncertainty_margin : {best_fb['uncertainty_margin']:.6f}")
print(f"  fallback_rate      : {best_fb['fallback_rate']:.4f}")
print(f"  AUC                : {best_fb['auc']:.4f}" if not np.isnan(best_fb["auc"]) else "  AUC                : nan")
print(f"  F1_macro           : {best_fb['f1_macro']:.4f}")
print(f"  F1_yes             : {best_fb['f1_yes']:.4f}")

# Guardar predicciones finales usando la mejor configuración con fallback
final_eval = evaluate(
    model=model,
    criterion=SoftLabelLoss(),
    loader=test_loader,
    device=device,
    save_path=test_save_path,
    decision_threshold=best_fb["threshold"],
    uncertainty_margin=best_fb["uncertainty_margin"],
    fallback_to_qwen=True,
    return_outputs=True,
 )

print("\nPredicciones finales guardadas en:", test_save_path)

# 12.1 Evaluar ajustando threshold

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 13 · Cargar checkpoint ya entrenado + evaluación por etapas (inteligente)
# ══════════════════════════════════════════════════════════════════════════════

# pretrained_model_path = "../data/processed/xlm-roberta-base-73.3.pt"
pretrained_model_path = "../data/processed/xlm-roberta-base.pt"
assert os.path.exists(pretrained_model_path), f"No existe el checkpoint: {pretrained_model_path}"

# Reconstruir arquitectura y cargar pesos
loaded_model = MultimodalModel(
    eeg_dim=eeg_dim,
    et_hr_dim=et_hr_dim,
    text_dim=TEXT_DIM,
    num_heads=8,
    freeze_backbone=False,
).to(device)

state_dict = torch.load(pretrained_model_path, map_location=device)
loaded_model.load_state_dict(state_dict, strict=True)
loaded_model.eval()
print(f"Checkpoint cargado correctamente desde: {pretrained_model_path}")

# Asegurar test loader con qwen para fallback (si no lo corriste en la celda 12)
if "test_loader" not in globals():
    with open(os.path.join(DATA_DIR, "test.json"), encoding="utf-8") as f:
        test_data = json.load(f)

    test_dataset = MemeDataset(
        test_data, tokenizer,
        eeg_dim=eeg_dim, et_hr_dim=et_hr_dim,
        include_qwen=True,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=32, shuffle=False,
        collate_fn=collate_fn, num_workers=4, pin_memory=True,
    )

# 1) Obtener probs del modelo cargado con baseline estricto (0.5, sin fallback)
eval_loaded_base = evaluate(
    model=loaded_model,
    criterion=SoftLabelLoss(),
    loader=test_loader,
    device=device,
    save_path=None,
    decision_threshold=0.50,
    uncertainty_margin=0.0,
    fallback_to_qwen=False,
    return_outputs=True,
)

# 2) Ajuste por etapas con candidatos inteligentes (sin grids fijos)
stage_loaded = staged_threshold_tuning(
    probs=eval_loaded_base["probs"],
    labels=eval_loaded_base["labels"],
    qwen_preds=eval_loaded_base["qwen_preds"],
    threshold_grid=None,
    uncertainty_grid=None,
    sort_by="f1_macro",
)

baseline_loaded = stage_loaded["baseline"]
best_no_fb_loaded = stage_loaded["best_no_fallback"]
best_fb_loaded = stage_loaded["best_with_fallback"]

print("\n" + "═" * 68)
print("  LOADED MODEL · STAGE 1 (baseline 0.50, sin fallback)")
print("═" * 68)
print(f"  AUC      : {baseline_loaded['auc']:.2%}" if not np.isnan(baseline_loaded["auc"]) else "  AUC      : nan")
print(f"  F1_macro : {baseline_loaded['f1_macro']:.2%}")
print(f"  F1_yes   : {baseline_loaded['f1_yes']:.2%}")

print("\n" + "═" * 68)
print("  LOADED MODEL · STAGE 2 (mejor threshold sin fallback, midpoints)")
print("═" * 68)
print(f"  threshold          : {best_no_fb_loaded['threshold']:.4%}")
print(f"  uncertainty_margin : {best_no_fb_loaded['uncertainty_margin']:.4%}")
print(f"  AUC                : {best_no_fb_loaded['auc']:.2%}" if not np.isnan(best_no_fb_loaded["auc"]) else "  AUC                : nan")
print(f"  F1_macro           : {best_no_fb_loaded['f1_macro']:.2%}")
print(f"  F1_yes             : {best_no_fb_loaded['f1_yes']:.2%}")

print("\n" + "═" * 68)
print("  LOADED MODEL · STAGE 3 (mejor uncertainty con fallback, midpoints)")
print("═" * 68)
print(f"  threshold          : {best_fb_loaded['threshold']:.4%}")
print(f"  uncertainty_margin : {best_fb_loaded['uncertainty_margin']:.4%}")
print(f"  fallback_rate      : {best_fb_loaded['fallback_rate']:.2%}")
print(f"  AUC                : {best_fb_loaded['auc']:.2%}" if not np.isnan(best_fb_loaded["auc"]) else "  AUC                : nan")
print(f"  F1_macro           : {best_fb_loaded['f1_macro']:.2%}")
print(f"  F1_yes             : {best_fb_loaded['f1_yes']:.2%}")

# Guardar predicciones finales del modelo cargado con la mejor configuración
loaded_eval_save_path = os.path.join(PREDS_DIR, f"{TEXT_ENCODER_NAME}_test_loaded_model.json")
final_loaded = evaluate(
    model=loaded_model,
    criterion=SoftLabelLoss(),
    loader=test_loader,
    device=device,
    save_path=loaded_eval_save_path,
    decision_threshold=best_fb_loaded["threshold"],
    uncertainty_margin=best_fb_loaded["uncertainty_margin"],
    fallback_to_qwen=True,
    return_outputs=True,
)

print("\nPredicciones finales del modelo cargado guardadas en:", loaded_eval_save_path)